In [1]:
from kafka import KafkaConsumer
import xml.etree.ElementTree as ET
import csv

BROKER = 'cld-kfk-04.brusnika.ltd'
TOPIC = 'bru.revit-plugin.logs'
CSV_FILE = 'output.csv'

# URI из вашего XML
NS_URI = 'http://schema.brusnika.tech/mdm/oa/pc/design/revit-plugin/revit-plugin-logs'
# тэги в XML (обратите внимание на дефисы)
TAGS = {
    'event': 'event',
    'username': 'username',
    'is_success': 'is-success',            # mapping: is-success -> is_success
    'assembly_timestamp': 'assembly-timestamp',
    'timestamp': 'timestamp'
}

def get_text(elem, tag):
    child = elem.find(f'{{{NS_URI}}}{tag}')
    return child.text if child is not None else ''

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=[BROKER],
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda m: m.decode('utf-8', errors='ignore'),
    consumer_timeout_ms=10000  # остановится если нет сообщений
)

with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['event','username','is_success','assembly_timestamp','timestamp'])

    for msg in consumer:
        xml = msg.value.strip()
        try:
            root = ET.fromstring(xml)
        except ET.ParseError:
            # можно логировать ошибку и продолжать
            continue

        row = []
        for col, tag in TAGS.items():
            row.append(get_text(root, tag))
        writer.writerow(row)

consumer.close()